In [ ]:
import sys
from pyprojroot import here

# Ritorna il percorso assoluto della root del progetto
PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)
sys.dont_write_bytecode = True

print(f"La root del progetto è: {PROJECT_ROOT}")

In [ ]:
from paths import INV_PATH, TP1_PATH

ABS_PATH = PROJECT_ROOT + INV_PATH + TP1_PATH
LOAD_MSH = PROJECT_ROOT

model_name = "rock"
xdmf_file_name = "rock.xdmf"

# Nome figure
fig_dir = "figures/"
eig_fig = ABS_PATH + fig_dir + "sv.png"
error_fig = ABS_PATH + fig_dir + "error.png"
speed_up_fig = ABS_PATH + fig_dir + "speed_up.png"
samples_fig = ABS_PATH + fig_dir + "samples.png"

dbdir = ABS_PATH + "trainPOD/"

In [ ]:
import fenics as fe

from fem_problems.invTP1.finite_element import PoissonFEM
from fem_problems.invTP1.rbnics_pod import PODReduction

Carichiamo la mesh

In [ ]:
# Path to the XDMF file
file_path = ABS_PATH + xdmf_file_name

# Carica la mesh da file XDMF
mesh = fe.Mesh()
with fe.XDMFFile(file_path) as infile:
    infile.read(mesh)

Definiamo il range dei parametri

In [ ]:
mu_range = [
    (0, 1.),
    (0, 1.),
    (0, 1.),
]

Definiamo il problema fem e il pod

In [ ]:
fem_p = PoissonFEM(mesh)
pod = PODReduction(mu_range, fem_p)

In [ ]:
num_training = 100
num_testing = 10

N_modes = num_training
tol = 1e-12

In [ ]:
training = pod.sampling_parameters(num_training)
testing = pod.sampling_parameters(num_testing, test=True)
pod.plot_samples(figsize = (6, 6), filename=samples_fig)

In [ ]:
pod.pod_execution(N_modes, tol)

In [ ]:
pod.plot_eigenvalues(filename=eig_fig)

In [ ]:
pod.store_reduction(directory=dbdir, filename=model_name)

In [ ]:
mu_test = [.2, .6, .4]
mat, _ = pod.solve_rom(mu_test, pod.num_basis)
fom_m, _ = pod.fem_p.solve_fem(mu_test)
real_m, _ = pod.fem_p.exact_solution(mu_test)

print("ERRORE FOM-ESATTA")
e, f = pod.fem_p.compute_error(fom_m, real_m)
print("ERRORE ROM-FOM")
a, b = pod.fem_p.compute_error(mat, fom_m)
print("ERRORE ROM-ESATTA")
c, d = pod.fem_p.compute_error(mat, real_m)

In [ ]:
error, _ = pod.error_analysis()

In [ ]:
pod.plot_errors(error, filename=error_fig)

In [ ]:
pod.plot_speed_up(error, filename=speed_up_fig)

In [ ]:
pod.save_table_error(error)

In [ ]:
pod.table_error(error)